# Personalized Artist Recommendation with LightGCN

DSCI 351 Recommender Systems, Group 16.

LightGCN is trained with BPR loss and compared with Most Popular, UserKNN, ItemKNN, BPR-MF, and WRMF on the HetRec 2011 Last.fm 2K dataset.

All models use the same five per-user splits, warm-start catalog, full-catalog candidate set, and Top-10 metrics.


## 1. Environment and data

The official archive is stored under `data/raw/`. If it is missing, the next cell downloads it from GroupLens and extracts it. The interaction file contains one row per observed `(user, artist)` pair and a listening count named `weight`.

Observed pairs are treated as positive implicit feedback. Listening counts are used only by WRMF as confidence; all ranking metrics use binary relevance.

In [ ]:
from pathlib import Path
import urllib.request
import zipfile

PROJECT_DIR = Path.cwd().resolve()
DATA_DIR = PROJECT_DIR / "data" / "raw" / "lastfm-2k"
ARCHIVE = PROJECT_DIR / "data" / "raw" / "hetrec2011-lastfm-2k.zip"
DATA_URL = "https://files.grouplens.org/datasets/hetrec2011/hetrec2011-lastfm-2k.zip"

if not (DATA_DIR / "user_artists.dat").exists():
    ARCHIVE.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(DATA_URL, ARCHIVE)
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ARCHIVE) as zf:
        zf.extractall(DATA_DIR)

print(f"Dataset ready: {(DATA_DIR / 'user_artists.dat').exists()}")

## 2. Reproducible implementation

The cell below contains every model and evaluation component used in the experiment:

- per-user 80/10/10 splitting with a warm-start catalog constraint;
- full-catalog ranking with already-seen artists removed;
- Recall@10, nDCG@10, MRR@10, Catalog Coverage@10, and Novelty@10;
- Most Popular, UserKNN, ItemKNN, BPR-MF, WRMF-ALS, and LightGCN;
- validation-based early stopping for BPR-MF and LightGCN.


In [ ]:
from __future__ import annotations

import argparse
import json
import math
import random
import time
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Callable, Dict, Iterable, List, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
import torch.nn.functional as F
from sklearn.metrics.pairwise import cosine_similarity


@dataclass(frozen=True)
class Config:
    top_k: int = 10
    user_knn_k: int = 40
    item_knn_k: int = 120
    factors: int = 64
    wrmf_factors: int = 16
    wrmf_alpha: float = 5.0
    wrmf_reg: float = 0.10
    wrmf_iterations: int = 5
    bpr_lr: float = 0.01
    bpr_reg: float = 1e-4
    bpr_epochs: int = 300
    lightgcn_lr: float = 0.01
    lightgcn_reg: float = 1e-4
    lightgcn_epochs: int = 800
    lightgcn_layers: int = 2
    eval_every: int = 50
    patience_checks: int = 4


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def load_lastfm(data_dir: Path) -> Tuple[pd.DataFrame, pd.DataFrame, Dict[int, int], Dict[int, int]]:
    interactions = pd.read_csv(data_dir / "user_artists.dat", sep="\t")
    artists = pd.read_csv(data_dir / "artists.dat", sep="\t", encoding="latin-1")
    user_ids = np.sort(interactions["userID"].unique())
    item_ids = np.sort(interactions["artistID"].unique())
    user_map = {int(raw): int(idx) for idx, raw in enumerate(user_ids)}
    item_map = {int(raw): int(idx) for idx, raw in enumerate(item_ids)}
    interactions = interactions.copy()
    interactions["user"] = interactions["userID"].map(user_map).astype(np.int64)
    interactions["item"] = interactions["artistID"].map(item_map).astype(np.int64)
    return interactions, artists, user_map, item_map


def split_per_user(
    interactions: pd.DataFrame,
    n_users: int,
    n_items: int,
    seed: int,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, Dict[str, int]]:
    """80/10/10 per-user split with warm-start catalog repair.

    Each user with at least three interactions contributes at least one validation and
    one test record. If an item would otherwise be absent from training, one of its
    held-out occurrences is moved into training. This prevents impossible cold-item
    test cases without looking at listening-count magnitudes.
    """
    rng = np.random.default_rng(seed)
    train_idx: List[int] = []
    val_idx: List[int] = []
    test_idx: List[int] = []

    for _, group in interactions.groupby("user", sort=False):
        idx = group.index.to_numpy(copy=True)
        rng.shuffle(idx)
        n = len(idx)
        if n < 3:
            train_idx.extend(idx.tolist())
            continue
        n_val = max(1, int(round(0.10 * n)))
        n_test = max(1, int(round(0.10 * n)))
        while n - n_val - n_test < 1:
            if n_val >= n_test and n_val > 1:
                n_val -= 1
            elif n_test > 1:
                n_test -= 1
            else:
                break
        val_idx.extend(idx[:n_val].tolist())
        test_idx.extend(idx[n_val : n_val + n_test].tolist())
        train_idx.extend(idx[n_val + n_test :].tolist())

    train_set = set(train_idx)
    val_set = set(val_idx)
    test_set = set(test_idx)
    train_items = set(interactions.loc[list(train_set), "item"].astype(int))
    moved_to_train = 0
    for item, group in interactions.groupby("item", sort=False):
        if int(item) in train_items:
            continue
        candidates = [int(i) for i in group.index if int(i) in val_set or int(i) in test_set]
        if not candidates:
            continue
        chosen = candidates[int(rng.integers(len(candidates)))]
        if chosen in val_set:
            val_set.remove(chosen)
        else:
            test_set.remove(chosen)
        train_set.add(chosen)
        train_items.add(int(item))
        moved_to_train += 1

    train = interactions.loc[sorted(train_set)].reset_index(drop=True)
    val = interactions.loc[sorted(val_set)].reset_index(drop=True)
    test = interactions.loc[sorted(test_set)].reset_index(drop=True)
    stats = {
        "train_pairs": int(len(train)),
        "validation_pairs": int(len(val)),
        "test_pairs": int(len(test)),
        "catalog_repairs": int(moved_to_train),
        "test_users": int(test["user"].nunique()),
        "catalog_items": int(n_items),
    }
    return train, val, test, stats


def make_csr(frame: pd.DataFrame, n_users: int, n_items: int, weighted: bool = False) -> sp.csr_matrix:
    if weighted:
        values = frame["weight"].to_numpy(np.float32)
    else:
        values = np.ones(len(frame), dtype=np.float32)
    return sp.csr_matrix(
        (values, (frame["user"].to_numpy(), frame["item"].to_numpy())),
        shape=(n_users, n_items),
        dtype=np.float32,
    )


def truth_lists(frame: pd.DataFrame, n_users: int) -> List[np.ndarray]:
    grouped = frame.groupby("user")["item"].apply(lambda x: x.to_numpy(np.int64)).to_dict()
    return [grouped.get(u, np.empty(0, dtype=np.int64)) for u in range(n_users)]


def topk_from_scores(scores: np.ndarray, seen: sp.csr_matrix, k: int) -> np.ndarray:
    scores = np.asarray(scores, dtype=np.float32)
    for row in range(scores.shape[0]):
        start, end = seen.indptr[row], seen.indptr[row + 1]
        scores[row, seen.indices[start:end]] = -np.inf
    candidate = np.argpartition(-scores, kth=k - 1, axis=1)[:, :k]
    values = np.take_along_axis(scores, candidate, axis=1)
    order = np.argsort(-values, axis=1)
    return np.take_along_axis(candidate, order, axis=1).astype(np.int64)


def rank_all_users(
    score_block: Callable[[np.ndarray], np.ndarray],
    seen: sp.csr_matrix,
    n_users: int,
    k: int,
    block_size: int = 256,
) -> np.ndarray:
    recs = np.empty((n_users, k), dtype=np.int64)
    for start in range(0, n_users, block_size):
        stop = min(start + block_size, n_users)
        rows = np.arange(start, stop)
        block = score_block(rows)
        recs[start:stop] = topk_from_scores(block, seen[start:stop], k)
    return recs


def evaluate_recommendations(
    recs: np.ndarray,
    truth: Sequence[np.ndarray],
    train_popularity: np.ndarray,
    n_users: int,
    n_items: int,
) -> Dict[str, float]:
    recalls: List[float] = []
    ndcgs: List[float] = []
    reciprocal_ranks: List[float] = []
    valid_recs: List[np.ndarray] = []
    for u, relevant in enumerate(truth):
        if len(relevant) == 0:
            continue
        rel_set = set(int(x) for x in relevant)
        row = recs[u]
        hits = np.array([1.0 if int(i) in rel_set else 0.0 for i in row], dtype=np.float64)
        recalls.append(float(hits.sum() / len(rel_set)))
        discounts = 1.0 / np.log2(np.arange(2, len(row) + 2))
        dcg = float((hits * discounts).sum())
        idcg = float(discounts[: min(len(rel_set), len(row))].sum())
        ndcgs.append(dcg / idcg if idcg else 0.0)
        hit_positions = np.flatnonzero(hits)
        reciprocal_ranks.append(1.0 / float(hit_positions[0] + 1) if len(hit_positions) else 0.0)
        valid_recs.append(row)
    flat = np.concatenate(valid_recs) if valid_recs else np.empty(0, dtype=np.int64)
    coverage = float(np.unique(flat).size / n_items) if len(flat) else 0.0
    popularity_probability = (train_popularity + 1.0) / (n_users + 1.0)
    novelty = float(np.mean(-np.log2(popularity_probability[flat]))) if len(flat) else 0.0
    return {
        "Recall@10": float(np.mean(recalls)),
        "nDCG@10": float(np.mean(ndcgs)),
        "MRR@10": float(np.mean(reciprocal_ranks)),
        "Coverage@10": coverage,
        "Novelty@10": novelty,
        "Evaluated users": float(len(recalls)),
    }


def prune_topk(matrix: sp.csr_matrix, k: int) -> sp.csr_matrix:
    matrix = matrix.tocsr()
    indptr = [0]
    indices: List[int] = []
    values: List[float] = []
    for row in range(matrix.shape[0]):
        start, end = matrix.indptr[row], matrix.indptr[row + 1]
        row_values = matrix.data[start:end]
        row_indices = matrix.indices[start:end]
        if len(row_values) > k:
            keep = np.argpartition(row_values, -k)[-k:]
            order = keep[np.argsort(-row_values[keep])]
        else:
            order = np.argsort(-row_values)
        indices.extend(row_indices[order].tolist())
        values.extend(row_values[order].tolist())
        indptr.append(len(indices))
    return sp.csr_matrix(
        (np.asarray(values, dtype=np.float32), np.asarray(indices, dtype=np.int32), np.asarray(indptr, dtype=np.int64)),
        shape=matrix.shape,
    )


def fit_userknn(train: sp.csr_matrix, k: int) -> sp.csr_matrix:
    similarities = cosine_similarity(train, dense_output=False).tocsr().astype(np.float32)
    similarities.setdiag(0.0)
    similarities.eliminate_zeros()
    return prune_topk(similarities, k)


def fit_itemknn(train: sp.csr_matrix, k: int) -> sp.csr_matrix:
    similarities = cosine_similarity(train.T, dense_output=False).tocsr().astype(np.float32)
    similarities.setdiag(0.0)
    similarities.eliminate_zeros()
    return prune_topk(similarities, k)


def sample_negatives(users: np.ndarray, positives: np.ndarray, train: sp.csr_matrix, rng: np.random.Generator) -> np.ndarray:
    negatives = rng.integers(0, train.shape[1], size=len(users), dtype=np.int64)
    invalid = np.asarray(train[users, negatives]).reshape(-1) > 0
    while invalid.any():
        negatives[invalid] = rng.integers(0, train.shape[1], size=int(invalid.sum()), dtype=np.int64)
        invalid = np.asarray(train[users, negatives]).reshape(-1) > 0
    return negatives


class BPRMF(torch.nn.Module):
    def __init__(self, n_users: int, n_items: int, factors: int) -> None:
        super().__init__()
        self.user = torch.nn.Embedding(n_users, factors)
        self.item = torch.nn.Embedding(n_items, factors)
        torch.nn.init.normal_(self.user.weight, std=0.05)
        torch.nn.init.normal_(self.item.weight, std=0.05)


def train_bprmf(
    train: sp.csr_matrix,
    truth_val: Sequence[np.ndarray],
    cfg: Config,
    seed: int,
    device: torch.device,
) -> Tuple[np.ndarray, np.ndarray, Dict[str, float]]:
    set_seed(seed)
    coo = train.tocoo()
    users_np = coo.row.astype(np.int64)
    items_np = coo.col.astype(np.int64)
    model = BPRMF(train.shape[0], train.shape[1], cfg.factors).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.bpr_lr)
    rng = np.random.default_rng(seed + 101)
    users = torch.as_tensor(users_np, dtype=torch.long, device=device)
    pos = torch.as_tensor(items_np, dtype=torch.long, device=device)
    best_metric = -np.inf
    best_state = None
    best_epoch = 0
    checks_without_gain = 0
    pop = np.asarray(train.sum(axis=0)).reshape(-1)
    for epoch in range(1, cfg.bpr_epochs + 1):
        negatives_np = sample_negatives(users_np, items_np, train, rng)
        neg = torch.as_tensor(negatives_np, dtype=torch.long, device=device)
        pos_score = (model.user(users) * model.item(pos)).sum(dim=1)
        neg_score = (model.user(users) * model.item(neg)).sum(dim=1)
        loss = F.softplus(-(pos_score - neg_score)).mean()
        reg = (
            model.user(users).pow(2).sum(dim=1)
            + model.item(pos).pow(2).sum(dim=1)
            + model.item(neg).pow(2).sum(dim=1)
        ).mean()
        loss = loss + cfg.bpr_reg * reg
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if epoch % cfg.eval_every == 0 or epoch == cfg.bpr_epochs:
            user_f = model.user.weight.detach().cpu().numpy()
            item_f = model.item.weight.detach().cpu().numpy()
            recs = rank_all_users(lambda rows: user_f[rows] @ item_f.T, train, train.shape[0], cfg.top_k)
            score = evaluate_recommendations(recs, truth_val, pop, train.shape[0], train.shape[1])["nDCG@10"]
            if score > best_metric + 1e-6:
                best_metric = score
                best_epoch = epoch
                checks_without_gain = 0
                best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            else:
                checks_without_gain += 1
                if checks_without_gain >= cfg.patience_checks:
                    break
    if best_state is not None:
        model.load_state_dict(best_state)
    return (
        model.user.weight.detach().cpu().numpy().astype(np.float32),
        model.item.weight.detach().cpu().numpy().astype(np.float32),
        {"best_validation_nDCG@10": float(best_metric), "best_epoch": float(best_epoch)},
    )


def normalized_bipartite_adjacency(train: sp.csr_matrix, device: torch.device) -> torch.Tensor:
    coo = train.tocoo()
    n_users, n_items = train.shape
    user_degree = np.asarray(train.sum(axis=1)).reshape(-1)
    item_degree = np.asarray(train.sum(axis=0)).reshape(-1)
    values = 1.0 / np.sqrt(user_degree[coo.row] * item_degree[coo.col])
    rows = np.concatenate([coo.row, coo.col + n_users])
    cols = np.concatenate([coo.col + n_users, coo.row])
    values = np.concatenate([values, values]).astype(np.float32)
    indices = torch.as_tensor(np.vstack([rows, cols]), dtype=torch.long, device=device)
    vals = torch.as_tensor(values, dtype=torch.float32, device=device)
    return torch.sparse_coo_tensor(indices, vals, (n_users + n_items, n_users + n_items), device=device).coalesce()


class LightGCN(torch.nn.Module):
    def __init__(self, n_users: int, n_items: int, factors: int, layers: int, adjacency: torch.Tensor) -> None:
        super().__init__()
        self.n_users = n_users
        self.n_items = n_items
        self.layers = layers
        self.adjacency = adjacency
        self.embedding = torch.nn.Embedding(n_users + n_items, factors)
        torch.nn.init.normal_(self.embedding.weight, std=0.05)

    def propagate(self) -> Tuple[torch.Tensor, torch.Tensor]:
        current = self.embedding.weight
        layers = [current]
        for _ in range(self.layers):
            current = torch.sparse.mm(self.adjacency, current)
            layers.append(current)
        final = torch.stack(layers, dim=0).mean(dim=0)
        return final[: self.n_users], final[self.n_users :]


def train_lightgcn(
    train: sp.csr_matrix,
    truth_val: Sequence[np.ndarray],
    cfg: Config,
    seed: int,
    device: torch.device,
) -> Tuple[np.ndarray, np.ndarray, Dict[str, float]]:
    set_seed(seed)
    coo = train.tocoo()
    users_np = coo.row.astype(np.int64)
    items_np = coo.col.astype(np.int64)
    adjacency = normalized_bipartite_adjacency(train, device)
    model = LightGCN(train.shape[0], train.shape[1], cfg.factors, cfg.lightgcn_layers, adjacency).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lightgcn_lr)
    rng = np.random.default_rng(seed + 303)
    users = torch.as_tensor(users_np, dtype=torch.long, device=device)
    pos = torch.as_tensor(items_np, dtype=torch.long, device=device)
    pop = np.asarray(train.sum(axis=0)).reshape(-1)
    best_metric = -np.inf
    best_state = None
    best_epoch = 0
    checks_without_gain = 0
    for epoch in range(1, cfg.lightgcn_epochs + 1):
        negatives_np = sample_negatives(users_np, items_np, train, rng)
        neg = torch.as_tensor(negatives_np, dtype=torch.long, device=device)
        user_f, item_f = model.propagate()
        pos_score = (user_f[users] * item_f[pos]).sum(dim=1)
        neg_score = (user_f[users] * item_f[neg]).sum(dim=1)
        ranking_loss = F.softplus(-(pos_score - neg_score)).mean()
        raw_u = model.embedding(users)
        raw_p = model.embedding(pos + train.shape[0])
        raw_n = model.embedding(neg + train.shape[0])
        reg = (raw_u.pow(2).sum(dim=1) + raw_p.pow(2).sum(dim=1) + raw_n.pow(2).sum(dim=1)).mean()
        loss = ranking_loss + cfg.lightgcn_reg * reg
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if epoch % cfg.eval_every == 0 or epoch == cfg.lightgcn_epochs:
            with torch.no_grad():
                val_u, val_i = model.propagate()
                u_np = val_u.detach().cpu().numpy()
                i_np = val_i.detach().cpu().numpy()
            recs = rank_all_users(lambda rows: u_np[rows] @ i_np.T, train, train.shape[0], cfg.top_k)
            score = evaluate_recommendations(recs, truth_val, pop, train.shape[0], train.shape[1])["nDCG@10"]
            if score > best_metric + 1e-6:
                best_metric = score
                best_epoch = epoch
                checks_without_gain = 0
                best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            else:
                checks_without_gain += 1
                if checks_without_gain >= cfg.patience_checks:
                    break
    if best_state is not None:
        model.load_state_dict(best_state)
    with torch.no_grad():
        final_u, final_i = model.propagate()
    return (
        final_u.detach().cpu().numpy().astype(np.float32),
        final_i.detach().cpu().numpy().astype(np.float32),
        {"best_validation_nDCG@10": float(best_metric), "best_epoch": float(best_epoch)},
    )


def fit_wrmf_als(
    weighted_train: sp.csr_matrix,
    factors: int,
    alpha: float,
    reg: float,
    iterations: int,
    seed: int,
) -> Tuple[np.ndarray, np.ndarray]:
    """Confidence-weighted implicit MF using the Hu-Koren-Volinsky ALS updates."""
    rng = np.random.default_rng(seed)
    n_users, n_items = weighted_train.shape
    x = rng.normal(0, 0.05, size=(n_users, factors)).astype(np.float64)
    y = rng.normal(0, 0.05, size=(n_items, factors)).astype(np.float64)
    confidence = weighted_train.copy().astype(np.float64)
    confidence.data = 1.0 + alpha * np.log1p(confidence.data)
    confidence_t = confidence.T.tocsr()
    eye = np.eye(factors, dtype=np.float64)

    def update(target: np.ndarray, fixed: np.ndarray, matrix: sp.csr_matrix) -> None:
        gram = fixed.T @ fixed
        base = gram + reg * eye
        for row in range(matrix.shape[0]):
            start, end = matrix.indptr[row], matrix.indptr[row + 1]
            idx = matrix.indices[start:end]
            c = matrix.data[start:end]
            if len(idx) == 0:
                target[row] = 0.0
                continue
            f = fixed[idx]
            a = base + (f.T * (c - 1.0)) @ f
            b = f.T @ c
            target[row] = np.linalg.solve(a, b)

    for _ in range(iterations):
        update(x, y, confidence)
        update(y, x, confidence_t)
    return x.astype(np.float32), y.astype(np.float32)


def retrain_pairwise_model(
    kind: str,
    fit: sp.csr_matrix,
    cfg: Config,
    seed: int,
    device: torch.device,
    epochs: int,
) -> Tuple[np.ndarray, np.ndarray]:
    set_seed(seed)
    coo = fit.tocoo()
    users_np = coo.row.astype(np.int64)
    items_np = coo.col.astype(np.int64)
    users = torch.as_tensor(users_np, dtype=torch.long, device=device)
    pos = torch.as_tensor(items_np, dtype=torch.long, device=device)
    rng = np.random.default_rng(seed + 707)
    if kind == "BPR-MF":
        model = BPRMF(fit.shape[0], fit.shape[1], cfg.factors).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=cfg.bpr_lr)
        for _ in range(epochs):
            neg_np = sample_negatives(users_np, items_np, fit, rng)
            neg = torch.as_tensor(neg_np, dtype=torch.long, device=device)
            pos_score = (model.user(users) * model.item(pos)).sum(dim=1)
            neg_score = (model.user(users) * model.item(neg)).sum(dim=1)
            loss = F.softplus(-(pos_score - neg_score)).mean()
            reg = (
                model.user(users).pow(2).sum(dim=1)
                + model.item(pos).pow(2).sum(dim=1)
                + model.item(neg).pow(2).sum(dim=1)
            ).mean()
            loss = loss + cfg.bpr_reg * reg
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        return (
            model.user.weight.detach().cpu().numpy().astype(np.float32),
            model.item.weight.detach().cpu().numpy().astype(np.float32),
        )

    adjacency = normalized_bipartite_adjacency(fit, device)
    model = LightGCN(fit.shape[0], fit.shape[1], cfg.factors, cfg.lightgcn_layers, adjacency).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lightgcn_lr)
    for _ in range(epochs):
        neg_np = sample_negatives(users_np, items_np, fit, rng)
        neg = torch.as_tensor(neg_np, dtype=torch.long, device=device)
        user_f, item_f = model.propagate()
        pos_score = (user_f[users] * item_f[pos]).sum(dim=1)
        neg_score = (user_f[users] * item_f[neg]).sum(dim=1)
        ranking_loss = F.softplus(-(pos_score - neg_score)).mean()
        raw_u = model.embedding(users)
        raw_p = model.embedding(pos + fit.shape[0])
        raw_n = model.embedding(neg + fit.shape[0])
        reg = (raw_u.pow(2).sum(dim=1) + raw_p.pow(2).sum(dim=1) + raw_n.pow(2).sum(dim=1)).mean()
        loss = ranking_loss + cfg.lightgcn_reg * reg
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    with torch.no_grad():
        final_u, final_i = model.propagate()
    return final_u.detach().cpu().numpy().astype(np.float32), final_i.detach().cpu().numpy().astype(np.float32)


def evaluate_one_seed(
    interactions: pd.DataFrame,
    n_users: int,
    n_items: int,
    cfg: Config,
    seed: int,
    device: torch.device,
) -> Tuple[List[Dict[str, float]], Dict[str, object]]:
    start = time.time()
    train_df, val_df, test_df, split_stats = split_per_user(interactions, n_users, n_items, seed)
    train = make_csr(train_df, n_users, n_items)
    fit_df = pd.concat([train_df, val_df], ignore_index=True)
    fit = make_csr(fit_df, n_users, n_items)
    fit_weighted = make_csr(fit_df, n_users, n_items, weighted=True)
    val_truth = truth_lists(val_df, n_users)
    test_truth = truth_lists(test_df, n_users)
    pop = np.asarray(fit.sum(axis=0)).reshape(-1)
    rows_all = np.arange(n_users)
    results: List[Dict[str, float]] = []
    diagnostics: Dict[str, object] = {"seed": seed, "split": split_stats}

    def record(model: str, recs: np.ndarray, extra: Dict[str, float] | None = None) -> None:
        metrics = evaluate_recommendations(recs, test_truth, pop, n_users, n_items)
        row: Dict[str, float] = {"Seed": float(seed), "Model": model}  # type: ignore[assignment]
        row.update(metrics)
        results.append(row)
        if extra:
            diagnostics[model] = extra

    popularity = pop.astype(np.float32)
    recs = rank_all_users(lambda rows: np.repeat(popularity[None, :], len(rows), axis=0), fit, n_users, cfg.top_k)
    record("Most Popular", recs)

    user_sim = fit_userknn(train, cfg.user_knn_k)
    val_scores = user_sim @ train
    user_val_recs = rank_all_users(lambda rows: val_scores[rows].toarray(), train, n_users, cfg.top_k)
    diagnostics["UserKNN_validation_nDCG@10"] = evaluate_recommendations(
        user_val_recs, val_truth, np.asarray(train.sum(axis=0)).reshape(-1), n_users, n_items
    )["nDCG@10"]
    user_sim = fit_userknn(fit, cfg.user_knn_k)
    user_scores = user_sim @ fit
    recs = rank_all_users(lambda rows: user_scores[rows].toarray(), fit, n_users, cfg.top_k)
    record("UserKNN", recs)
    del user_scores, user_sim, val_scores

    item_sim = fit_itemknn(train, cfg.item_knn_k)
    val_scores = train @ item_sim
    item_val_recs = rank_all_users(lambda rows: val_scores[rows].toarray(), train, n_users, cfg.top_k)
    diagnostics["ItemKNN_validation_nDCG@10"] = evaluate_recommendations(
        item_val_recs, val_truth, np.asarray(train.sum(axis=0)).reshape(-1), n_users, n_items
    )["nDCG@10"]
    item_sim = fit_itemknn(fit, cfg.item_knn_k)
    item_scores = fit @ item_sim
    recs = rank_all_users(lambda rows: item_scores[rows].toarray(), fit, n_users, cfg.top_k)
    record("ItemKNN", recs)
    del item_scores, item_sim, val_scores

    bpr_u, bpr_i, bpr_diag = train_bprmf(train, val_truth, cfg, seed, device)
    best_bpr_epoch = max(cfg.eval_every, int(bpr_diag["best_epoch"]))
    bpr_u, bpr_i = retrain_pairwise_model("BPR-MF", fit, cfg, seed + 1000, device, best_bpr_epoch)
    recs = rank_all_users(lambda rows: bpr_u[rows] @ bpr_i.T, fit, n_users, cfg.top_k)
    record("BPR-MF", recs, bpr_diag)

    wrmf_u, wrmf_i = fit_wrmf_als(
        fit_weighted,
        cfg.wrmf_factors,
        cfg.wrmf_alpha,
        cfg.wrmf_reg,
        cfg.wrmf_iterations,
        seed,
    )
    recs = rank_all_users(lambda rows: wrmf_u[rows] @ wrmf_i.T, fit, n_users, cfg.top_k)
    record("WRMF", recs)

    lg_u, lg_i, lg_diag = train_lightgcn(train, val_truth, cfg, seed, device)
    best_lg_epoch = max(cfg.eval_every, int(lg_diag["best_epoch"]))
    lg_u, lg_i = retrain_pairwise_model("LightGCN", fit, cfg, seed + 2000, device, best_lg_epoch)
    recs = rank_all_users(lambda rows: lg_u[rows] @ lg_i.T, fit, n_users, cfg.top_k)
    record("LightGCN", recs, lg_diag)

    diagnostics["elapsed_seconds"] = float(time.time() - start)
    return results, diagnostics


def make_summary(results: pd.DataFrame) -> pd.DataFrame:
    metrics = ["Recall@10", "nDCG@10", "MRR@10", "Coverage@10", "Novelty@10"]
    grouped = results.groupby("Model")[metrics].agg(["mean", "std"])
    grouped.columns = [f"{metric}_{stat}" for metric, stat in grouped.columns]
    summary = grouped.reset_index()
    order = ["Most Popular", "UserKNN", "ItemKNN", "BPR-MF", "WRMF", "LightGCN"]
    summary["order"] = summary["Model"].map({name: i for i, name in enumerate(order)})
    return summary.sort_values("order").drop(columns="order").reset_index(drop=True)


def plot_results(summary: pd.DataFrame, figures_dir: Path) -> None:
    figures_dir.mkdir(parents=True, exist_ok=True)
    colors = ["#C8B6A6", "#A8B6A3", "#8FA8B8", "#D39B78", "#967A67", "#2E6F73"]
    plt.rcParams.update({
        "font.family": "DejaVu Sans",
        "font.size": 10,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.titleweight": "bold",
        "figure.facecolor": "#F7F1E7",
        "axes.facecolor": "#F7F1E7",
    })
    models = summary["Model"].tolist()
    x = np.arange(len(models))

    fig, axes = plt.subplots(1, 3, figsize=(13.2, 4.1), constrained_layout=True)
    for ax, metric in zip(axes, ["Recall@10", "nDCG@10", "MRR@10"]):
        mean = summary[f"{metric}_mean"].to_numpy()
        std = summary[f"{metric}_std"].fillna(0).to_numpy()
        bars = ax.bar(x, mean, yerr=std, capsize=3, color=colors, edgecolor="#5D534A", linewidth=0.5)
        ax.set_title(metric)
        ax.set_xticks(x, models, rotation=28, ha="right")
        ax.grid(axis="y", alpha=0.22)
        ax.set_ylim(0, max(mean + std) * 1.28)
        for bar, value in zip(bars, mean):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + max(mean) * 0.035, f"{value:.3f}", ha="center", va="bottom", fontsize=8)
    fig.suptitle("Top-10 ranking accuracy across five random splits", fontsize=14, fontweight="bold", color="#3A2E28")
    fig.savefig(figures_dir / "ranking_accuracy.png", dpi=220, bbox_inches="tight")
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(7.6, 5.0), constrained_layout=True)
    for idx, row in summary.iterrows():
        ax.scatter(row["Coverage@10_mean"], row["nDCG@10_mean"], s=170, color=colors[idx], edgecolor="#4B4038", linewidth=0.7, zorder=3)
        ax.annotate(row["Model"], (row["Coverage@10_mean"], row["nDCG@10_mean"]), xytext=(7, 5), textcoords="offset points", fontsize=9)
    ax.set_xlabel("Catalog Coverage@10")
    ax.set_ylabel("nDCG@10")
    ax.set_title("Accuracy–coverage trade-off")
    ax.grid(alpha=0.22)
    fig.savefig(figures_dir / "accuracy_coverage.png", dpi=220, bbox_inches="tight")
    plt.close(fig)



In [ ]:
interactions, artists, user_map, item_map = load_lastfm(DATA_DIR)
n_users = int(interactions["user"].nunique())
n_items = int(interactions["item"].nunique())

dataset_summary = pd.DataFrame({
    "Statistic": ["Users", "Artists", "Observed pairs", "Matrix density", "Mean artists per user", "Median listeners per artist"],
    "Value": [
        f"{n_users:,}",
        f"{n_items:,}",
        f"{len(interactions):,}",
        f"{len(interactions) / (n_users * n_items):.3%}",
        f"{interactions.groupby('user').size().mean():.2f}",
        f"{interactions.groupby('item').size().median():.0f}",
    ],
})
dataset_summary

## 3. Model selection on validation data

Only validation nDCG@10 was used to choose the main hyperparameters. The stored table below records the search used for the project. The final configuration is:

- UserKNN: 40 neighbors
- ItemKNN: 120 neighbors
- BPR-MF: 64 factors, learning rate 0.01
- WRMF: 16 factors, alpha 5, regularization 0.1, 5 ALS iterations
- LightGCN: 64 factors, 2 propagation layers, learning rate 0.01; early stopping up to 800 epochs

The test partitions were not used during this choice.

In [ ]:
# Set to True to rerun validation search instead of reading the saved table.
RUN_VALIDATION_SEARCH = False

def validation_search(seed=42):
    train_df, val_df, _, _ = split_per_user(interactions, n_users, n_items, seed)
    train = make_csr(train_df, n_users, n_items)
    weighted = make_csr(train_df, n_users, n_items, weighted=True)
    truth = truth_lists(val_df, n_users)
    pop = np.asarray(train.sum(axis=0)).reshape(-1)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    rows = []

    for model_name, fitter, candidates in [
        ("UserKNN", fit_userknn, [40, 80, 120]),
        ("ItemKNN", fit_itemknn, [40, 80, 120]),
    ]:
        for k in candidates:
            sim = fitter(train, k)
            scores = sim @ train if model_name == "UserKNN" else train @ sim
            recs = rank_all_users(lambda r: scores[r].toarray(), train, n_users, 10)
            ndcg = evaluate_recommendations(recs, truth, pop, n_users, n_items)["nDCG@10"]
            rows.append({"Model": model_name, "Configuration": f"k={k}", "Validation nDCG@10": ndcg})

    for layers, factors, lr, epochs in [(1, 32, .01, 240), (2, 32, .01, 240), (3, 32, .01, 240), (2, 64, .01, 800)]:
        cfg = Config(factors=factors, lightgcn_layers=layers, lightgcn_lr=lr,
                     lightgcn_epochs=epochs, eval_every=50, patience_checks=4)
        _, _, diag = train_lightgcn(train, truth, cfg, seed, device)
        rows.append({"Model": "LightGCN", "Configuration": f"L={layers}, d={factors}, lr={lr:g}",
                     "Validation nDCG@10": diag["best_validation_nDCG@10"], "Best epoch": diag["best_epoch"]})

    for factors, lr in [(32, .01), (64, .01), (64, .005)]:
        cfg = Config(factors=factors, bpr_lr=lr, bpr_epochs=300, eval_every=20, patience_checks=4)
        _, _, diag = train_bprmf(train, truth, cfg, seed, device)
        rows.append({"Model": "BPR-MF", "Configuration": f"d={factors}, lr={lr:g}",
                     "Validation nDCG@10": diag["best_validation_nDCG@10"], "Best epoch": diag["best_epoch"]})

    for factors, alpha in [(16, 5), (16, 10), (16, 20), (32, 10)]:
        u, i = fit_wrmf_als(weighted, factors, alpha, .1, 5, seed)
        recs = rank_all_users(lambda r: u[r] @ i.T, train, n_users, 10)
        ndcg = evaluate_recommendations(recs, truth, pop, n_users, n_items)["nDCG@10"]
        rows.append({"Model": "WRMF", "Configuration": f"d={factors}, alpha={alpha}, reg=0.1",
                     "Validation nDCG@10": ndcg})
    return pd.DataFrame(rows)

search_path = PROJECT_DIR / "results" / "hyperparameter_search.csv"
if RUN_VALIDATION_SEARCH:
    search_results = validation_search()
    search_results.to_csv(search_path, index=False)
else:
    search_results = pd.read_csv(search_path)

search_results.sort_values(["Model", "Validation nDCG@10"], ascending=[True, False]).groupby("Model").head(3).reset_index(drop=True)

## 4. Five-seed test experiment

For each seed, all six methods receive the same split. Hyperparameters and early stopping use validation data only. The selected epoch count is then used to retrain BPR-MF and LightGCN from scratch on `train + validation`, followed by one test evaluation.

The candidate set is the full 17,632-artist catalog minus artists already seen by that user in the fitting data.

In [ ]:
SEEDS = [42, 43, 44, 45, 46]
cfg = Config()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__}; device={device}")

all_rows = []
all_diagnostics = []
for seed in SEEDS:
    print(f"Running seed {seed} ...", flush=True)
    rows, diagnostics = evaluate_one_seed(interactions, n_users, n_items, cfg, seed, device)
    all_rows.extend(rows)
    all_diagnostics.append(diagnostics)
    print(f"  completed in {diagnostics['elapsed_seconds']:.1f} seconds", flush=True)

results = pd.DataFrame(all_rows)
summary = make_summary(results)
(PROJECT_DIR / "results").mkdir(parents=True, exist_ok=True)
(PROJECT_DIR / "figures").mkdir(parents=True, exist_ok=True)
results.to_csv(PROJECT_DIR / "results" / "metrics_by_seed.csv", index=False)
summary.to_csv(PROJECT_DIR / "results" / "metrics_summary.csv", index=False)
with open(PROJECT_DIR / "results" / "run_metadata.json", "w", encoding="utf-8") as handle:
    json.dump({"config": asdict(cfg), "seeds": SEEDS, "device": str(device),
               "dataset": {"users": n_users, "artists": n_items, "pairs": len(interactions)},
               "diagnostics": all_diagnostics}, handle, indent=2)
plot_results(summary, PROJECT_DIR / "figures")
summary

## 5. Results and interpretation

The primary metric is nDCG@10. Mean and sample standard deviation are computed over the five matched splits.

In [ ]:
display_table = summary[[
    "Model", "Recall@10_mean", "Recall@10_std", "nDCG@10_mean", "nDCG@10_std",
    "MRR@10_mean", "MRR@10_std", "Coverage@10_mean", "Novelty@10_mean"
]].copy()
display_table.columns = [
    "Model", "Recall mean", "Recall sd", "nDCG mean", "nDCG sd",
    "MRR mean", "MRR sd", "Coverage", "Novelty"
]
display_table.style.format({column: "{:.4f}" for column in display_table.columns if column != "Model"}).highlight_max(
    subset=["Recall mean", "nDCG mean", "MRR mean", "Coverage", "Novelty"], color="#d8eadf"
)

In [ ]:
pivot = results.pivot(index="Seed", columns="Model", values="nDCG@10")
lightgcn = summary.loc[summary["Model"] == "LightGCN"].iloc[0]
baselines = summary.loc[summary["Model"] != "LightGCN"]

analysis = {
    "LightGCN nDCG@10": lightgcn["nDCG@10_mean"],
    "Relative nDCG gain over strongest baseline": lightgcn["nDCG@10_mean"] / baselines["nDCG@10_mean"].max() - 1,
    "Relative Recall gain over strongest baseline": lightgcn["Recall@10_mean"] / baselines["Recall@10_mean"].max() - 1,
    "Relative MRR gain over strongest baseline": lightgcn["MRR@10_mean"] / baselines["MRR@10_mean"].max() - 1,
    "Seeds won by LightGCN": int((pivot.idxmax(axis=1) == "LightGCN").sum()),
}
pd.Series(analysis).to_frame("Value")

### Additional plots

The next cell reads the saved data and results to plot the artist popularity distribution, nDCG leaderboard, and accuracy-coverage trade-off. It saves PNG and PDF versions under `figures/`.


In [ ]:
# Plot saved data and results.
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display


FIGURE_DIR = PROJECT_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

summary_plot = pd.read_csv(PROJECT_DIR / "results" / "metrics_summary.csv")
interactions_plot = pd.read_csv(
    PROJECT_DIR / "data" / "raw" / "lastfm-2k" / "user_artists.dat",
    sep="\t",
)

INK = "#332B27"
ACCENT = "#D55E00"      # Okabe-Ito vermillion: LightGCN
BLUE = "#0072B2"        # Okabe-Ito blue: strongest accuracy baseline
SKY = "#56B4E9"
GOLD = "#E69F00"        # Okabe-Ito orange: coverage leader
GRAY = "#A79D94"
GRID = "#D9D0C6"
PAPER = "#F7F1E7"

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif"],
    "font.size": 11,
    "axes.titlesize": 14,
    "axes.labelsize": 11,
    "axes.titleweight": "bold",
    "axes.edgecolor": INK,
    "axes.labelcolor": INK,
    "xtick.color": INK,
    "ytick.color": INK,
    "text.color": INK,
    "figure.facecolor": PAPER,
    "axes.facecolor": PAPER,
    "savefig.facecolor": PAPER,
    "savefig.bbox": "tight",
})


def finish_axis(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis="both", color=GRID, linewidth=0.7, alpha=0.65)
    ax.set_axisbelow(True)


# Figure 1: dataset long tail
artist_popularity = (
    interactions_plot.groupby("artistID")["userID"]
    .nunique()
    .sort_values(ascending=False)
)
ranks = np.arange(1, len(artist_popularity) + 1)
single_listener_share = (artist_popularity == 1).mean()
top_one_percent_n = max(1, int(np.ceil(0.01 * len(artist_popularity))))
top_one_percent_share = artist_popularity.iloc[:top_one_percent_n].sum() / artist_popularity.sum()

fig, ax = plt.subplots(figsize=(6.4, 3.8))
ax.plot(ranks, artist_popularity.to_numpy(), color=ACCENT, linewidth=2.2)
ax.fill_between(ranks, artist_popularity.to_numpy(), 1, color=ACCENT, alpha=0.10)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Artist popularity rank (log scale)")
ax.set_ylabel("Distinct listeners (log scale)")
ax.set_title("Artist popularity follows a long tail", loc="left", pad=10)
finish_axis(ax)
ax.annotate(
    f"Top 1% of artists account for\n{top_one_percent_share:.1%} of observed pairs",
    xy=(top_one_percent_n, artist_popularity.iloc[top_one_percent_n - 1]),
    xytext=(18, 120),
    textcoords="data",
    fontsize=10,
    fontweight="bold",
    color=INK,
    arrowprops=dict(arrowstyle="->", color=INK, linewidth=1.1),
    bbox=dict(boxstyle="round,pad=0.35", facecolor=PAPER, edgecolor=GOLD, linewidth=1.0),
)
ax.annotate(
    f"{single_listener_share:.1%} of artists\nhave only one listener",
    xy=(len(artist_popularity), 1),
    xytext=(1250, 4.0),
    textcoords="data",
    fontsize=10,
    fontweight="bold",
    ha="center",
    color=INK,
    arrowprops=dict(arrowstyle="->", color=INK, linewidth=1.1),
    bbox=dict(boxstyle="round,pad=0.35", facecolor=PAPER, edgecolor=BLUE, linewidth=1.0),
)
fig.tight_layout()
for suffix, dpi in [("png", 300), ("pdf", 300)]:
    fig.savefig(FIGURE_DIR / f"dataset_long_tail_presentation.{suffix}", dpi=dpi)
plt.close(fig)


# Figure 2: primary ranking result
leaderboard = summary_plot.sort_values("nDCG@10_mean", ascending=False).reset_index(drop=True)
best_baseline_value = leaderboard.loc[leaderboard["Model"] != "LightGCN", "nDCG@10_mean"].max()
gain = leaderboard.loc[leaderboard["Model"] == "LightGCN", "nDCG@10_mean"].iloc[0] / best_baseline_value - 1
colors = []
for model in leaderboard["Model"]:
    if model == "LightGCN":
        colors.append(ACCENT)
    elif model == "UserKNN":
        colors.append(BLUE)
    else:
        colors.append(GRAY)

fig, ax = plt.subplots(figsize=(7.0, 4.3))
y = np.arange(len(leaderboard))
bars = ax.barh(
    y,
    leaderboard["nDCG@10_mean"],
    xerr=leaderboard["nDCG@10_std"],
    color=colors,
    edgecolor="none",
    error_kw=dict(ecolor=INK, elinewidth=1.0, capsize=3, capthick=1.0),
)
ax.set_yticks(y, leaderboard["Model"])
ax.invert_yaxis()
ax.set_xlabel("nDCG@10 (mean ± SD over five shared random splits)")
ax.set_title("LightGCN achieves the best ranking quality", loc="left", pad=10)
ax.set_xlim(0, 0.245)
ax.grid(axis="x", color=GRID, linewidth=0.8, alpha=0.75)
ax.set_axisbelow(True)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)
ax.tick_params(axis="y", length=0, pad=8)
for bar, mean, std in zip(bars, leaderboard["nDCG@10_mean"], leaderboard["nDCG@10_std"]):
    ax.text(mean + std + 0.004, bar.get_y() + bar.get_height() / 2, f"{mean:.3f}", va="center", fontsize=10)
ax.text(
    0.239,
    0.55,
    f"+{gain:.1%}\nvs. UserKNN",
    ha="right",
    va="center",
    fontsize=11,
    fontweight="bold",
    color=ACCENT,
)
fig.tight_layout()
for suffix, dpi in [("png", 300), ("pdf", 300)]:
    fig.savefig(FIGURE_DIR / f"ndcg_leaderboard_presentation.{suffix}", dpi=dpi)
plt.close(fig)


# Figure 3: accuracy–coverage trade-off
model_colors = {
    "LightGCN": ACCENT,
    "UserKNN": BLUE,
    "ItemKNN": GOLD,
    "Most Popular": GRAY,
    "BPR-MF": GRAY,
    "WRMF": GRAY,
}
label_offsets = {
    "LightGCN": (7, 7),
    "UserKNN": (7, 7),
    "ItemKNN": (-62, 8),
    "Most Popular": (7, 6),
    "BPR-MF": (7, -15),
    "WRMF": (7, -14),
}

fig, ax = plt.subplots(figsize=(5.0, 4.3))
for _, row in summary_plot.iterrows():
    model = row["Model"]
    x = 100 * row["Coverage@10_mean"]
    xerr = 100 * row["Coverage@10_std"]
    yval = row["nDCG@10_mean"]
    yerr = row["nDCG@10_std"]
    color = model_colors[model]
    ax.errorbar(
        x,
        yval,
        xerr=xerr,
        yerr=yerr,
        fmt="o",
        markersize=8 if model in {"LightGCN", "ItemKNN"} else 6.5,
        color=color,
        ecolor=color,
        elinewidth=1.0,
        capsize=2.5,
        markeredgecolor=PAPER,
        markeredgewidth=0.8,
        zorder=3,
    )
    dx, dy = label_offsets[model]
    ax.annotate(
        model,
        (x, yval),
        xytext=(dx, dy),
        textcoords="offset points",
        fontsize=9,
        fontweight="bold" if model in {"LightGCN", "ItemKNN"} else "normal",
        color=INK,
    )

ax.set_xlim(-0.5, 20.3)
ax.set_ylim(0.055, 0.232)
ax.set_xlabel("Catalog coverage@10 (%)")
ax.set_ylabel("nDCG@10")
ax.set_title("Accuracy–coverage trade-off", loc="left", pad=27)
finish_axis(ax)
ax.text(
    0.0,
    1.01,
    "Mean ± SD over five shared random splits",
    transform=ax.transAxes,
    va="bottom",
    fontsize=9,
    color="#6F655D",
)
fig.tight_layout()
for suffix, dpi in [("png", 300), ("pdf", 300)]:
    fig.savefig(FIGURE_DIR / f"accuracy_coverage_presentation.{suffix}", dpi=dpi)
plt.close(fig)

print("Figures saved to:", FIGURE_DIR)
display(Image(filename=str(FIGURE_DIR / "dataset_long_tail_presentation.png"), width=800))
display(Image(filename=str(FIGURE_DIR / "ndcg_leaderboard_presentation.png"), width=850))
display(Image(filename=str(FIGURE_DIR / "accuracy_coverage_presentation.png"), width=700))


## 6. Conclusion

Across five matched random splits, **LightGCN ranked first on Recall@10, nDCG@10, and MRR@10 in every seed**. Its mean nDCG@10 is about **13.2% higher** than the strongest baseline, UserKNN. It also improves substantially over BPR-MF, indicating that neighborhood propagation on the user–artist graph adds useful collaborative signal beyond direct user/item embeddings.

The trade-off is catalog breadth: ItemKNN obtains the highest Coverage@10 and Novelty@10. Therefore, the project supports LightGCN as the best accuracy-oriented model, while also showing that a production system might add reranking or diversity constraints when discovery of long-tail artists is a primary goal.

**Limitations.** The dataset contains listening counts but no listening timestamps, so the experiment uses random per-user splits rather than temporal evaluation. Artists with a single observed listener cannot be evaluated as true cold-start items under a collaborative-only protocol.

## References

- Xiangnan He et al. (2020). [LightGCN: Simplifying and Powering Graph Convolution Network for Recommendation](https://arxiv.org/abs/2002.02126).
- Steffen Rendle et al. (2009). [BPR: Bayesian Personalized Ranking from Implicit Feedback](https://arxiv.org/abs/1205.2618).
- [HetRec 2011 datasets](https://grouplens.org/datasets/hetrec-2011/), Information Retrieval Group, Universidad Autonoma de Madrid. The Last.fm archive includes its original README and non-commercial data usage terms.
